In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter
import itertools
import pmdarima as pm
from IPython.display import display


from sklearn.preprocessing import StandardScaler

import yfinance as yf
from fredapi import Fred
import pandas as pd
import seaborn as sns
from scipy import stats


from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# Helper Functions

In [ ]:
# ********************************************************************************************************
# Helper Functions
# ********************************************************************************************************

def run_strategy(df, initial_capital, leverage):

    df_strat = df.copy()
    n = len(df_strat)

    theta = np.zeros(n)  # Dollar position in the risky asset

    V = np.zeros(n) # Risky asset portfolio portion
    dV = np.zeros(n) # Daily PnL from risky asset

    V_cap = np.zeros(n)  # Cash portion (money-market)
    dV_cap = np.zeros(n)  # Daily PnL from cash

    V_tot = np.zeros(n)  # Total portfolio value
    dV_tot = np.zeros(n)  # Total daily PnL

    # ---- Initialization ----
    V_tot[0] = initial_capital      # Total starting capital
    V_cap[0] = initial_capital     # Initially, all capital is cash (no risky exposure)
    V[0] = 0                # No allocation in the risky asset at the start

    theta[0] = df_strat['final_signal'].iloc[0]*V_tot[0]*leverage


    for t in range(1, n):
        risky_position = theta[t-1] # Dollar-Value of the position in the risky asset

        # Daily PnL of Risky asset  & Update of its value
        dV[t] = df_strat['excess_returns'].iloc[t]*theta[t-1]
        V[t] = V[t-1] + dV[t]

        # Daily PnL of the Cash Position & Update of its value
        M_t = abs(risky_position)/leverage
        dV_cap[t] = (V_tot[t-1] - M_t)*df_strat['rf_daily'].iloc[t] 
        V_cap[t] = V_cap[t-1] + dV_cap[t]

        # Total Daily PnL & Update of the total portfolio value
        dV_tot[t] = dV[t] + dV_cap[t]
        V_tot[t] = V_cap[t] + V[t]

        # Update position in risky asset for the next trading day
        theta[t] = df_strat['final_signal'].iloc[t]*V_tot[t]*leverage

    strategy_returns = {
        'V': V,
        'dV': dV,
        'V_cap': V_cap,
        'dV_cap': dV_cap,
        'V_tot': V_tot,
        'dV_tot': dV_tot,
        'theta': theta
    }

    return strategy_returns



# PERFORMANCE METRICS

def compute_performance_metrics(strategy_dict):
    # Daily Returns
    total_returns = strategy_dict['dV_tot'][1:]/strategy_dict['V_tot'][:-1]

    # Sharpe Ratio
    expected_total_return = np.mean(total_returns)
    vol_total_return = np.std(total_returns)

    sharpe_ratio = (expected_total_return * 252) / (vol_total_return * np.sqrt(252)) if vol_total_return > 0 else np.nan


    # Maximum Drawdown
    daily_total_pnl = strategy_dict['dV_tot'][1:]
    cum_total_pnl = np.cumsum(daily_total_pnl)

    peak_value = np.maximum.accumulate(cum_total_pnl)
    # print(peak_value)
    drawdown = (cum_total_pnl - peak_value) / peak_value

    max_drawdown = np.min(drawdown)

    # Calmar Ratio
    if max_drawdown < 0:
        calmar_ratio = - 1 * (expected_total_return * 252) / max_drawdown
    else: 
        calmar_ratio = np.nan

    performance_metrics = {'Sharpe': sharpe_ratio,
            'Max Drawdown': max_drawdown,
            'Calmar Ratio': calmar_ratio,
            'Total Return': cum_total_pnl[-1]}

    return performance_metrics


def plot_in_out_daily_pnl(in_results, out_results, title_prefix="Strategy"):
    
    # time axes
    n_ins = len(in_results['dV_tot'])
    t_ins = np.arange(n_ins)
    t_oos = n_ins + np.arange(len(out_results['dV_tot']))

    fig, ax = plt.subplots(2, 1, figsize=(10, 8), sharex=False)
    
    # In-Sample Daily PNLs
    ax[0].plot(t_ins, in_results['dV'],     label='Risky Asset - Daily PnL', color='tab:blue', lw=0.8)
    ax[0].plot(t_ins, in_results['dV_cap'], label='Cash - Daily PnL', color='tab:green', lw=0.8)
    ax[0].plot(t_ins, in_results['dV_tot'], label='Total Daily PnL', color='tab:red', 
               lw=0.8, marker='o', markersize=5, markevery=[0, -1], markerfacecolor='tab:red',
               markeredgecolor='none')
    ax[0].set_title("In-Sample")
    ax[0].set_ylabel("PnL ($)")
    ax[0].legend(loc='upper left')
    ax[0].grid(alpha=0.3)

    # Out of Sample Daily PNLs
    ax[1].plot(t_oos, out_results['dV'],     label='Risky Asset - Daily PnL', color='tab:blue', lw=0.8)
    ax[1].plot(t_oos, out_results['dV_cap'], label='Cash - Daily PnL', color='tab:green', lw=0.8)
    ax[1].plot(t_oos, out_results['dV_tot'], label='Total Daily PnL', color='tab:red', lw=0.8,
               marker='o', markersize=5, markevery=[0, -1], markerfacecolor='tab:red', markeredgecolor='none')
    ax[1].set_title("Out-of-Sample")
    ax[1].set_xlabel("Time (days)")
    ax[1].set_ylabel("PnL ($)")
    ax[1].grid(alpha=0.3)

    for a in ax:
        fmt = ScalarFormatter(useOffset=False)
        fmt.set_scientific(False)
        a.yaxis.set_major_formatter(fmt)

    plt.tight_layout()
    plt.show()


def plot_in_out_cum_pnl(in_results, out_results, title_prefix="Strategy"):

    # time axes
    n_ins = len(in_results['dV_tot'])
    t_ins = np.arange(n_ins)
    t_oos = n_ins + np.arange(len(out_results['dV_tot']))

    # cumulative sums
    cum_ins_risky = np.cumsum(in_results['dV'])
    cum_ins_cash  = np.cumsum(in_results['dV_cap'])
    cum_ins_tot   = np.cumsum(in_results['dV_tot'])

    cum_oos_risky = np.cumsum(out_results['dV'])
    cum_oos_cash  = np.cumsum(out_results['dV_cap'])
    cum_oos_tot   = np.cumsum(out_results['dV_tot'])

    # create figure + axes
    fig, ax = plt.subplots(2, 1, figsize=(10, 8), sharex=False)

    # In‑sample cumulative PnL
    ax[0].plot(t_ins, cum_ins_risky, label='Risky Asset - Cum PnL', color='tab:blue', lw=0.8)
    ax[0].plot(t_ins, cum_ins_cash,  label='Cash - Cum PnL',  color='tab:green', lw=0.8)
    ax[0].plot(t_ins, cum_ins_tot,   label='Total Cum PnL', color='tab:red', lw=0.8,
               marker='o',markersize=5, markevery=[0, -1], markerfacecolor='tab:red', markeredgecolor='none')
    ax[0].set_title("In-Sample")
    ax[0].set_ylabel("Cumulative PnL ($)")
    ax[0].legend(loc='upper left')
    ax[0].grid(alpha=0.3)

    # Out‑of‑sample cumulative PnL
    ax[1].plot(t_oos, cum_oos_risky, label='Risky', color='tab:blue', lw=0.8)
    ax[1].plot(t_oos, cum_oos_cash,  label='Cash',  color='tab:green', lw=0.8)
    ax[1].plot(t_oos, cum_oos_tot,   label='Total', color='tab:red', lw=0.8,
               marker='o', markersize=5, markevery=[0, -1], markerfacecolor='tab:red', markeredgecolor='none')
    ax[1].set_title("Out-of-Sample")
    ax[1].set_xlabel("Time (Days)")
    ax[1].set_ylabel("Cumulative PnL ($)")
    ax[1].grid(alpha=0.3)

    for a in ax:
        fmt = ScalarFormatter(useOffset=False)
        fmt.set_scientific(False)
        a.yaxis.set_major_formatter(fmt)

    plt.tight_layout()
    plt.show()


def plot_train_test_cumulative_pnl(in_results, oos_results):
    """
    Plots the cumulative total PnL for both train and test sets, side by side.
    """
    # Build the cumulative sum of daily total PnL
    cum_pnl_train = np.cumsum(in_results['dV_tot'])
    cum_pnl_test  = np.cumsum(oos_results['dV_tot'])
    
    fig, axes = plt.subplots(2, 1, figsize=(10, 8), sharex=False)

    # ----- Top subplot: Training cumulative PnL -----
    axes[0].plot(cum_pnl_train, label='Train Cumulative PnL', color='blue')
    axes[0].set_title("Training Set: Cumulative Total PnL")
    axes[0].set_ylabel("PnL ($)")
    axes[0].grid(alpha=0.3)
    axes[0].legend()

    # ----- Bottom subplot: Testing cumulative PnL -----
    axes[1].plot(cum_pnl_test, label='Test Cumulative PnL', color='orange')
    axes[1].set_title("Testing Set: Cumulative Total PnL")
    axes[1].set_ylabel("PnL ($)")
    axes[1].set_xlabel("Days in Test Set")
    axes[1].grid(alpha=0.3)
    axes[1].legend()

    plt.tight_layout()
    plt.show()

# Initial Data Processing - Main Variables
1) Data retrieval - WARNING: Replace "your_fred_api_key_here" with actual api key
2) Stationarity Analysis of the main variable for coherent ARIMAX implementation

In [ ]:
# ********************************************************************************************************
# Data Preprocessing 
# ********************************************************************************************************

# ********************************************************************************************************
# Initial Data Preprocessing 
# ********************************************************************************************************

start_date = "2022-12-29"
end_date = "2023-12-31"

ticker = "SPTL"

# SPTL-ETF DATA
sptl = yf.download(ticker, start=start_date, end=end_date)
sptl.index = pd.to_datetime(sptl.index, format='%Y-%m-%d')
sptl.columns = sptl.columns.droplevel(1)
sptl.rename_axis(None, axis=1, inplace=True)
sptl.drop(columns=["Open"], inplace=True)

# Initital Price --> For ARIMAX signal generation
initial_price = sptl['Close'].iloc[0]

# EFFR DATA
fred = Fred(api_key="your_fred_api_key_here")  # Replace with your actual FRED API key
effr_series = fred.get_series("EFFR", observation_start=start_date, observation_end=end_date)
effr_data = pd.DataFrame(effr_series, columns=["rf_annual"])
effr_data["rf_annual"] = (effr_data["rf_annual"] / 100.0)
effr_data.index = pd.to_datetime(effr_data.index, format='%Y-%d-%m')
effr_data["rf_annual"] = effr_data["rf_annual"].ffill()


# Merging SPTL-ETF and EFFR Data
merged_df = pd.merge(sptl, effr_data, left_index=True, right_index=True, how='left')

# Creating Daily EFFR Rate - Taken as the Daily Risk-free Rate
merged_df['rf_daily'] = merged_df['rf_annual'] / 252.0

# Daily Returns & Excess SPTL Returns
merged_df['returns'] = merged_df['Close'].pct_change()
merged_df['diff_returns'] = merged_df['Close'].diff()
merged_df['excess_returns'] = merged_df['returns'] - merged_df['rf_daily']

merged_df.dropna(inplace=True)
merged_df.drop(['High', 'Low', 'Volume'], axis=1, inplace=True)

# STATIONARITY ANALYSIS

# Retrieving Data for Checks
df = merged_df.copy()

prices = df['Close'].to_numpy()
diff_returns = df['diff_returns'].to_numpy()
returns = df['returns'].to_numpy()

# Price Stationarity ADF Test
result_prices = adfuller(prices)
print("Price Series ADF Statistic:", result_prices[0])
print("Price Series p-value:", result_prices[1])
print("*"*120)

# Price Series - ACF & PACF plots

plt.figure(figsize=(12, 6))
plt.subplot(121)
plot_acf(prices, lags=30, ax=plt.gca())
plt.title('ACF - SPTL Closing Prices')

plt.subplot(122)
plot_pacf(prices, lags=30, ax=plt.gca(), method='ywm')
plt.title('PACF - SPTL Closing Prices')

plt.tight_layout()
plt.show()

# Returns ADF Test
result_returns = adfuller(returns)
print("Returns Series ADF Statistic:", result_returns[0])
print("Returns p-value:", result_returns[1])

# Price Series - ACF & PACF plots

plt.figure(figsize=(12, 6))
plt.subplot(121)
plot_acf(returns, lags=30, ax=plt.gca())
plt.title('ACF - SPTL Returns')

plt.subplot(122)
plot_pacf(returns, lags=30, ax=plt.gca(), method='ywm')
plt.title('PACF - SPTL Returns')

plt.tight_layout()
plt.show()

# Initial Data Processing - Exogenous Variables

1) Description of exogenous variable candidates in table below:
<div>
<style scoped>
    .dataframe tbody tr th:only-of-type {
        vertical-align: middle;
    }

    .dataframe tbody tr th {
        vertical-align: top;
    }

    .dataframe thead th {
        text-align: right;
    }
</style>
<table border="1" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th>Variable</th>
      <th>Description</th>
      <th>Series ID / Ticker</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th>0</th>
      <td>Y5</td>
      <td>5‑Year U.S. Treasury yield</td>
      <td>DGS5</td>
    </tr>
    <tr>
      <th>1</th>
      <td>Y10</td>
      <td>10‑Year U.S. Treasury yield</td>
      <td>DGS10</td>
    </tr>
    <tr>
      <th>2</th>
      <td>Y30</td>
      <td>30‑Year U.S. Treasury yield</td>
      <td>DGS30</td>
    </tr>
    <tr>
      <th>3</th>
      <td>2Y10Y</td>
      <td>10‑Year minus 2‑Year yield spread</td>
      <td>T10Y2Y</td>
    </tr>
    <tr>
      <th>4</th>
      <td>5Y30Y</td>
      <td>30‑Year minus 5‑Year yield spread</td>
      <td>DGS30 – DGS5</td>
    </tr>
    <tr>
      <th>5</th>
      <td>t_premium</td>
      <td>Kim–Wright 10‑Year term premium</td>
      <td>THREEFYTP10</td>
    </tr>
    <tr>
      <th>6</th>
      <td>inflation</td>
      <td>10‑Year Breakeven Inflation Rate</td>
      <td>T10YIE</td>
    </tr>
    <tr>
      <th>7</th>
      <td>move_daily</td>
      <td>ICE BofA Merrill Lynch MOVE Index</td>
      <td>^MOVE</td>
    </tr>
  </tbody>
</table>
</div>

2) Data retrieval/cleaning of exogenous variables candidates - WARNING: Replace "your_fred_api_key_here" with actual fred api key!


3) Stationarity Analysis of these variables for coherent ARIMAX implementation

In [ ]:

# ********************************************************************************************************
# Data Preprocessing - Exog Variables - Additional Data for Exogenous Variables of Arimax
# ********************************************************************************************************

fred = Fred(api_key="your_fred_api_key_here")  # Replace with your actual FRED API key

spread_2y10 = (fred.get_series("T10Y2Y", observation_start=start_date, observation_end=end_date) / 100.0)
spread_2y10.name = "2Y10Y"

y5 = (fred.get_series("DGS5", observation_start=start_date, observation_end=end_date) / 100.0)
y5.name = "y5"

y10 = (fred.get_series("DGS10", observation_start=start_date, observation_end=end_date) / 100.0)
y10.name = "y10"

y30 = (fred.get_series("DGS30", observation_start=start_date, observation_end=end_date) / 100.0)
y30.name = "y30"

term_premium = fred.get_series("THREEFYTP10", observation_start=start_date, observation_end=end_date)
term_premium.name = "t_premium"

inflation_rate = (fred.get_series("T10YIE", observation_start=start_date, observation_end=end_date) / 100.0)
inflation_rate.name = "inflation"


move_series = yf.download("MOVE", start=start_date, end=end_date)
move_series.index = pd.to_datetime(move_series.index, format='%Y-%m-%d')
move_series.columns = move_series.columns.droplevel(1)
move_series.rename_axis(None, axis=1, inplace=True)
move_series.drop(columns=["Open", "High", "Low", "Volume"], inplace=True)
move_series.rename(columns={'Close': 'move_annual'}, inplace=True)
move_data = move_series.ffill()
move_data["move_annual"] = (move_data["move_annual"] / 100.0)

# creating Exogenous Vars Dataframe
dfs = [
    spread_2y10,
    y5,
    y10,
    y30,
    term_premium,
    inflation_rate,
    move_data
]

exog_df = pd.concat(dfs, axis=1, join="outer")
exog_df = exog_df.ffill()

exog_df['move_daily'] = (exog_df['move_annual'] / np.sqrt(252))
exog_df["5Y30Y"] = exog_df["y30"] - exog_df["y5"]

exog_df.drop(['move_annual'], axis=1, inplace=True)

# Check stationarity of Exogenous variables

def check_stationarity(df, alpha=0.05):
    rows = []
    for col in df.columns:
        stat, pval, *_ = adfuller(df[col].dropna(), autolag='AIC')
        rows.append({
            'series': col,
            'ADF Stat': round(stat, 3),
            'p_value': round(pval, 4),
            'stationary': pval < alpha
        })
    return pd.DataFrame(rows).set_index('series')


orig = check_stationarity(exog_df)

# Exogenous Variables (as well as the modelled - endogeneous one - must be stationary before going into ARIMAX)

# For Yields, Slopes, Term Premium and Breakeven inflation: First Difference Should be Enough Difference 
# For Volatility Indices: Log returns

exog_trans = exog_df.copy()
for col in ['y5', 'y10', 'y30', '2Y10Y', '5Y30Y','t_premium', 'inflation']:
    exog_trans[col] = exog_df[col].diff()


exog_trans["move_daily"] = np.log(exog_df["move_daily"]).diff()
exog_trans.dropna(inplace=True)

trans = check_stationarity(exog_trans)

display(orig)
display(trans)



Merging main variables with exogenous variables into one data structure

In [ ]:

exog_final = exog_trans.copy()
data = pd.merge(merged_df, exog_final, left_index=True, right_index=True, how="left")
final_data = data.copy()

cols_to_shift = [
     '2Y10Y', '5Y30Y','y5', 'y10', 'y30',
    't_premium', 'inflation',
    'move_daily'
]

final_data[cols_to_shift] = final_data[cols_to_shift].shift(1)
final_data = final_data.dropna()


# Train, Validation and Test Split

The worflow is as such:

1) We first split the data three ways into a training & validation set (in-sample data) as well as test set (out-of-sample unseen data)

2) We find the best ARIMA order using the AIC criterion selection on the training sample

3) Then keeping the ARIMA order fixed, we perform a grid search over different combinations of exogenous variables. For each candidate combination, we fit an ARIMAX model on the training set and generate one-step ahead forecasts over the validation set

4) We finally compare the resulting Sharpe Ratio and Maximum Drawdown over the validation set to select the best exogenous-variable combination.


# 1) Train, Validation and Test Split

In [ ]:
df = final_data.copy()
n = len(df)
train = int(n * 0.50)
test  = int(n * 0.80)

train_df = df.iloc[:train]
test_df  = df.iloc[train:test]
oos_df   = df.iloc[test:]  # keep for later


# 2) ARIMA Order Fitting
Evaluating best arima order on training sample using AIC Criterion Selection (NO EXOG)

In [ ]:
best_aic   = np.inf
best_order = None

for p, d, q in itertools.product(range(3), range(2), range(3)):
    try:
        tmp_mod = SARIMAX(
            train_df['returns'],
            order=(p, d, q),
            enforce_stationarity=False,
            enforce_invertibility=False
        )
        tmp_res = tmp_mod.fit(disp=False)
        if tmp_res.aic < best_aic:
            best_aic   = tmp_res.aic
            best_order = (p, d, q)
    except:
        pass

print(f"Selected ARIMAX order = {best_order} (AIC={best_aic:.1f})")

# 3) Exogenous variables grid search on validation set

In [ ]:
# WALK FORWARD - ONE STEP FORECASTING on the TEST set (i.e. validation set)

# 2) build feature‐sets
all_exog = [
    'y5', 'y10', 'y30',
    '2Y10Y', '5Y30Y',
    't_premium', 'inflation','move_daily'
]


feature_sets = []
for k in (1,2,3):
    feature_sets += list(itertools.combinations(all_exog, k))
results = []


# AUTOMATING FEATURE SELECTION
for exog_vars in feature_sets:

    exog_cols = list(exog_vars)
        # TRAIN SIGNAL
    train_prices = train_df['Close'].to_numpy()
    test_prices  = test_df['Close'].to_numpy()
    train_rets   = train_df['returns'].to_numpy()
    test_rets    = test_df['returns'].to_numpy()

    scaler     = StandardScaler().fit(train_df[exog_cols])
    exog_train = scaler.transform(train_df[exog_cols])


    model = SARIMAX(train_rets,
                    order=(1, 0, 0),
                    enforce_stationarity=False,
                    enforce_invertibility=False,
                    exog=exog_train).fit(disp=False)

    # Signal Generation Training Set
    trade_prices = np.empty(len(train_prices))
    trade_prices[0] = initial_price
    trade_prices[1:] = train_prices[:-1]

    rhat_train = model.fittedvalues
    phat_train = trade_prices * (1 + rhat_train)
    signal_train = np.sign(phat_train - trade_prices)

    train_sig_df = train_df.iloc[:len(signal_train)].copy()
    train_sig_df['final_signal'] = signal_train

    train_pnl = run_strategy(train_sig_df, initial_capital=100000.0, leverage=10.0)
    train_metrics = compute_performance_metrics(train_pnl)

    # TEST SIGNAL

    test_prices  = test_df['Close'].to_numpy()
    test_rets    = test_df['returns'].to_numpy()
    exog_test  = scaler.transform(test_df[exog_cols])

    history_rets = train_rets.tolist()
    history_exog = exog_train.tolist()

    rhat_test = np.empty(len(test_df))

    for i in range(len(test_df)):
        mdl = SARIMAX(
            history_rets,
            order=(1, 0, 0),
            exog=np.array(history_exog),
            enforce_stationarity=False,
            enforce_invertibility=False
        ).fit(disp=False)

        x_next = exog_test[i:i+1]
        rhat = mdl.forecast(steps=1, exog=x_next)[0]
        rhat_test[i] = rhat

        # Update window
        history_rets.append(test_rets[i])
        history_exog.append(x_next[0])


    test_trade_prices = np.empty_like(test_prices)
    test_trade_prices[0] = train_prices[-1]
    test_trade_prices[1:] = test_prices[:-1]

    phat_test = test_trade_prices * (1 + rhat_test)
    signal_test = np.sign(phat_test - test_trade_prices)

    test_sig_df = test_df.copy()
    test_sig_df['final_signal'] = signal_test

    test_pnl = run_strategy(test_sig_df, initial_capital=100000.0, leverage=10.0)
    test_metrics = compute_performance_metrics(test_pnl)

    results.append({
        "features": ", ".join(exog_vars),
        "train Sharpe": train_metrics["Sharpe"],
        "val Sharpe": test_metrics["Sharpe"],
        "train MaxDD": train_metrics.get("Max Drawdown", np.nan),
        "val MaxDD": test_metrics.get("Max Drawdown", np.nan)
    })


# 4) Grid search results
Resulting Sharpe Ratio and Maximum Drawdown for different exog variable configurations on the validation set

In [ ]:
# IN-SAMPLE PERFORMANCE --> OPTIMISED FOR HIGHEST SHARPE ON VALIDATION SET
results_df = pd.DataFrame(results).sort_values("val Sharpe", ascending=False)

# show top 10 only
display(results_df.head(5))

Final Model Selection Summary:

1) ARIMAX Order: (1, 0, 0)

2) Best exogenous features:
    a) 30-Year U.S. Treasury yield (y30)
    b) 10-Year U.S. Treasury yield Spread (2Y10Y)
    c) Kim-Wright 10-Year term premium (t_premium)

# Validating Final ARIMAX model residuals on the full in-sample dataset (training + validation)

In [ ]:

# ********************************************************************************************************
# Evaluate Final Model Residuals on FULL IN_SAMPLE TEST
# ********************************************************************************************************
full_in_sample = pd.concat([train_df, test_df])

# extract the 3 chosen exogs, scale them on in‑sample
chosen = ['y30','2Y10Y','t_premium']  
scaler = StandardScaler().fit(full_in_sample[chosen])
exog_in = scaler.transform(full_in_sample[chosen])

# fit  final ARIMAX(1,0,0) with chosen exogs
final_model = SARIMAX(
    full_in_sample['returns'],
    order=(1,0,0),
    exog=exog_in,
    enforce_stationarity=False,
    enforce_invertibility=False
)
final_model_results = final_model.fit(disp=False)
print(final_model_results.summary())

# Evaluate Final Model Residuals on FULL IN_SAMPLE TEST
residuals = final_model_results.resid
stdized_residuals = (residuals - np.mean(residuals)) / np.std(residuals)


import matplotlib.dates as mdates

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# 1) Standardized Residuals vs Time
ax = axes[0, 0]
ax.plot(stdized_residuals, label='Std Residuals')
ax.set_title("Standardized Residuals over full in-sample data")
ax.set_xlabel("Date")
ax.set_ylabel("Std Residuals")

# date‐formatting on the x-axis
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
fig.autofmt_xdate(rotation=30, ha='right')

# 2) Histogram + KDE with legend
ax = axes[0, 1]
sns.histplot(stdized_residuals, bins=15, stat="density",
             color="skyblue", label='Histogram', ax=ax)
sns.kdeplot(stdized_residuals, lw=2, label='KDE', ax=ax)
x_vals = np.linspace(-4, 4, 200)
ax.plot(x_vals, stats.norm.pdf(x_vals), '--r', label='Normal PDF')
ax.set_title("Histogram + KDE of Residuals")
ax.set_xlabel("Std Residual")
ax.legend()

# 3) Q-Q Plot
ax = axes[1, 0]
stats.probplot(stdized_residuals, dist="norm", plot=ax)
ax.set_title("Normal Q–Q Plot")

# 4) ACF Plot of Residuals
ax = axes[1, 1]
plot_acf(stdized_residuals, lags=30, ax=ax)
ax.set_title("ACF of Residuals")

plt.tight_layout()
plt.show()

The ARIMAX model residuals are validated

# Final Model Results 
We then obtain the performance metrics of the final ARIMAX model configuration on both the:
1) Full in-sample data (training & validation sets)
2) Full out-of-sample data (the held out and unseen test set)

Perfomance metrics include: (a) Sharpe Ratio, (b) Maximum Drawdown, c) Calmar Ratio, d) Total Return

The table is obtained after running cells 24 and 25 below.

In [ ]:
# Full-IN SAMPLE PERFORMANCE

train_val_df = pd.concat([train_df, test_df])


tv_prices = train_val_df['Close'].to_numpy()
oos_prices = oos_df['Close'].to_numpy()

tv_rets = train_val_df['returns'].to_numpy()
oos_rets = oos_df['returns'].to_numpy()

# 3) scale exogs on train+val, then transform oos
scaler    = StandardScaler().fit(train_val_df[chosen])
exog_tv   = scaler.transform(train_val_df[chosen])
exog_oos  = scaler.transform(oos_df[chosen])

in_model = SARIMAX(
    tv_rets,
    order=(1,0,0),
    exog=exog_tv,
    enforce_stationarity=False,
    enforce_invertibility=False
).fit(disp=False)

# 4b) Grab its in‑sample fitted values as the one‑step return forecasts
rhat_in = in_model.fittedvalues  # length = len(train_val_df)

# 4c) Build the trade‐price array for in-sample
trade_px_in = np.empty_like(tv_prices)
trade_px_in[0] = initial_price
trade_px_in[1:]   = tv_prices[:-1]
phat_in = trade_px_in * (1 + rhat_in)
signal_in = np.sign(phat_in - trade_px_in)

in_signal_df = train_val_df.copy()
in_signal_df['final_signal'] = signal_in


# Run Strategy
in_results = run_strategy(in_signal_df, initial_capital=100000.0, leverage=10.0)
in_metrics = compute_performance_metrics(in_results)



In [ ]:
# FULL OUT OF SAMPLE PERFORMANCE

history_rets = tv_rets.tolist()
history_exog = exog_tv.tolist()

#  roll one‑step ahead through oos_df
rhat_oos = np.empty(len(oos_df))
for i in range(len(oos_df)):
    mdl = SARIMAX(
        history_rets,
        exog=np.array(history_exog),
        order=(1, 0, 0),
        enforce_stationarity=False,
        enforce_invertibility=False
    ).fit(disp=False)

    x_next  = exog_oos[i:i+1]
    rhat_oos[i]  = mdl.forecast(steps=1, exog=x_next)[0]

    history_rets.append(oos_rets[i])
    history_exog.append(x_next[0])

# build trade‐price array for oos
oos_trade_px = np.empty_like(oos_prices)
oos_trade_px[0] = tv_prices[-1]       # last price of train+val
oos_trade_px[1:] = oos_prices[:-1]

# generate oos signals
phat_oos    = oos_trade_px * (1 + rhat_oos)
signal_oos  = np.sign(phat_oos - oos_trade_px)

#  attach and evaluate
oos_signal_df = oos_df.copy()
oos_signal_df['final_signal'] = signal_oos

oos_results = run_strategy(oos_signal_df, initial_capital=100_000.0, leverage=10.0)
oos_metrics = compute_performance_metrics(oos_results)


summary = pd.DataFrame({
    'Sharpe Ratio':   [in_metrics['Sharpe'],   oos_metrics['Sharpe']],
    'Max Drawdown':   [in_metrics['Max Drawdown'], oos_metrics['Max Drawdown']],
    'Calmar Ratio':   [in_metrics['Calmar Ratio'], oos_metrics['Calmar Ratio']],
    'Total Return':   [in_metrics['Total Return'], oos_metrics['Total Return']],
}, index=['In‑Sample','Out‑of‑Sample']).round(3)
display(summary)

# In-Sample vs Out-of-Sample Daily and Cummulative PnL performance plots 

These plots are naturally generated using the final model configuration of the ARIMAX model.

In [ ]:
# PLOTTING FINAL IN-SAMPLE vs OUT-SAMPLE Daily PnL Performance
plot_in_out_daily_pnl(in_results,  oos_results, title_prefix="ARIMAX Strategy PnL")

In [ ]:
# PLOTTING FINAL IN-SAMPLE vs OUT-SAMPLE Cummulative PnL Performance
plot_in_out_cum_pnl (in_results,  oos_results, title_prefix="ARIMAX Strategy PnL")